# Phase 37 -- full 60-question sanity check before commit/deploy

No LLM judge (that problem is still open, per Phase 31/CLAUDE.md) -- a regex-proxy
pass over the full `eval_v3` set, same methodology as every prior full-eval run in
this project, run under TODAY's complete production pipeline: Claude Haiku 4.5,
compact prompt, and every fix from this session (Phase 33's gender rank exception,
Phase 34's residual-question fixes, Phase 35's q09/q29 fixes, Phase 36's query
reparser -- now running on every question, unconditionally).

**Purpose:** the reparser touches every single answer, and it was only spot-checked
on 4 questions before now. This is the check before committing and deploying
anything -- catch a regression across the full set, not just the 4 questions
already looked at by hand.

**Baseline to compare against:** Phase 31's Haiku + compact-prompt re-review (same
model, same base prompt, none of today's fixes): recall@1 .883, recall@5 1.000, MRR
.926, full coverage .950 (retrieval, from Phase 24c/28, unchanged by generation-side
work); regex proxy 60/60 correct-article-cited, 58/60 fully complete, 0/60
fabrications.


In [ ]:
import sys
import json
from pathlib import Path

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv
load_dotenv(PROJECT / ".env")

import pandas as pd

from src.retrieval import Retriever
from src.generation import (get_production_generator, PRODUCTION_MODEL_ID,
                             extract_cited_articles, grounded_article_numbers)
from evallib.metrics import metrics

EVAL_DIR = PROJECT / "eval"
print("production model:", PRODUCTION_MODEL_ID)

eval_v3 = []
with open(EVAL_DIR / "eval_v3.jsonl", encoding="utf-8") as f:
    for line in f:
        eval_v3.append(json.loads(line))
print(f"{len(eval_v3)} questions")


In [ ]:
RESULTS_PATH = EVAL_DIR / "phase37_full_sanity_check.csv"

retriever = Retriever()
gen = get_production_generator()

rows = []
if RESULTS_PATH.exists():
    rows = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig").to_dict("records")
done = {r["question_id"] for r in rows}

for d in eval_v3:
    qid = d["question_id"]
    if qid in done:
        continue
    q = d["question_text"]
    gold_law = d["answer_law"]
    gold_arts = set(d["answer_article_numbers"])

    chunks = retriever.retrieve(q, top_k=5)
    retrieved_same_law = [c["article_number"] for c in chunks if c["law"] == gold_law]
    rank = next((i + 1 for i, a in enumerate(retrieved_same_law) if a in gold_arts), None)
    full_coverage = gold_arts.issubset(set(retrieved_same_law))

    res = gen.generate(q, chunks)
    answer = res["answer"]
    cited = extract_cited_articles(answer)
    grounded = grounded_article_numbers(chunks)
    fabricated = cited - grounded
    correct = bool(gold_arts & cited)
    all_correct = gold_arts.issubset(cited)

    rows.append({
        "question_id": qid, "question": q, "gold_law": gold_law,
        "gold_articles": sorted(gold_arts), "rank": rank, "full_coverage": full_coverage,
        "cited_articles": sorted(cited), "fabricated": sorted(fabricated),
        "correct": correct, "all_correct": all_correct,
        "answer_len": len(answer), "answer": answer,
    })
    pd.DataFrame(rows).to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")
    print(f"  {qid}: rank={rank} correct={correct} all_correct={all_correct} fabricated={sorted(fabricated)}")

results_df = pd.DataFrame(rows)
print(f"\n{len(results_df)}/{len(eval_v3)} done")


## Aggregate results

In [ ]:
print("=== Retrieval ===")
print(metrics(results_df, col="rank", ndigits=3, full_coverage=True))
print()
print("=== Regex proxy ===")
n = len(results_df)
print(f"correct-article-cited: {results_df['correct'].sum()}/{n}")
print(f"fully complete (all gold articles cited): {results_df['all_correct'].sum()}/{n}")
print(f"fabrications (cited but not grounded, any question): "
      f"{(results_df['fabricated'].apply(len) > 0).sum()}/{n}")
print(f"mean answer length: {results_df['answer_len'].mean():.0f} chars "
      f"(Phase 31 baseline, same config minus today's fixes: ~1,159 chars)")


In [ ]:
# Read every flagged case by hand before concluding anything -- a raised fabrication
# count or a lower correct-article rate needs reading, not just counting (Phase 20/21
# precedent: several "fabrication" flags turned out to be the article's own in-text
# cross-references, not real invented citations).
wrong = results_df[~results_df["correct"]]
incomplete = results_df[results_df["correct"] & ~results_df["all_correct"]]
flagged = results_df[results_df["fabricated"].apply(len) > 0]

print(f"{len(wrong)} wrong (gold article never cited):", list(wrong["question_id"]))
print(f"{len(incomplete)} partially correct (cited SOME but not all gold articles):", list(incomplete["question_id"]))
print(f"{len(flagged)} flagged fabrication candidates (need manual read):", list(flagged["question_id"]))


In [ ]:
for _, r in pd.concat([wrong, flagged]).drop_duplicates(subset="question_id").iterrows():
    print("=" * 90)
    print(r["question_id"], "-- gold:", r["gold_articles"], "| cited:", r["cited_articles"],
          "| fabricated flag:", r["fabricated"])
    print(r["question"])
    print()
    print(r["answer"][:1200])
    print()


## Verdict

*(compare the aggregate numbers above against the Phase 31 baseline stated at the
top; read every wrong/flagged case printed above and say, per case, whether it's a
real regression or a pre-existing/known issue; state plainly whether this clears the
bar to commit and deploy.)*
